# KAN-LoFTR fine-tuning on Kaggle (OHRC/NAC ↔ TMC-2)

Fine-tunes Kornia's pretrained LoFTR: **frozen ResNet-FPN backbone**, **trainable coarse attention**, and a **KAN** (`512 → 64 → 2`, G=5, k=3) that replaces LoFTR's fine stage and predicts a sub-pixel offset for each coarse match.

## Before you run (one-time setup)
1. **Add two Kaggle Datasets** (Create → New Dataset → upload the folder contents), then attach both to this notebook with *Add Input*:
   - `kan-loftr-code`: `kan.py, dataset.py, model.py, train.py, evaluate.py`
   - `lunar-pairs`: `patches_u8.npz` (and `index.json`)
2. **Settings → Accelerator → GPU** (T4 or P100). Only one GPU is used.
3. **Settings → Internet → On**. It is needed for `pip install kornia` if it is missing, and for Kornia to download the pretrained LoFTR weights (see cell 5 for the offline option).
4. *Run All*.

## Read this first
- The training pairs are only **assumed** aligned (the OHRC strip and the TMC swath have no measurable georeferenced overlap; `index.json` says so). Supervision comes from *synthetic* homographies applied to the stored pair, so the model learns to recover those warps. It does not prove real OHRC↔TMC registration.
- There are **36 train / 7 val / 7 test** patches of 128×128 px. That is very small: expect overfitting, and treat the numbers as a smoke test of the pipeline.
- The code has been syntax-checked but was **not executed with torch** before this notebook. Cell 4 is a fast smoke test that fails early with a readable error; if it fails, send the traceback.

In [ ]:
# 1. Environment
import sys, subprocess, importlib.util, shutil, json
from pathlib import Path
import torch

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU: enable an accelerator in Settings")
for pkg in ("kornia", "einops"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)   # needs Internet ON
import kornia
print("kornia", kornia.__version__)

In [ ]:
# 2. Locate the attached datasets and put the code on the path
INPUT, WORK = Path("/kaggle/input"), Path("/kaggle/working")

def find_one(name):
    hits = sorted(INPUT.rglob(name))
    if not hits:
        raise FileNotFoundError(f"'{name}' not found under /kaggle/input. Attach the dataset that contains it (see the header).")
    return hits[0]

CODE_DIR = find_one("model.py").parent
DATA = find_one("patches_u8.npz")
SRC, OUT = WORK / "src", WORK / "kan_loftr"
shutil.copytree(CODE_DIR, SRC, dirs_exist_ok=True)          # the input dataset is read-only
sys.path.insert(0, str(SRC))
print("code :", sorted(p.name for p in SRC.glob("*.py")))
print("data :", DATA, f"({DATA.stat().st_size / 1e6:.2f} MB)")

In [ ]:
# 3. Dataset check: split sizes and the guard band
!python {SRC}/dataset.py --check {DATA}

In [ ]:
# 4. SMOKE TEST (random weights, no downloads): KAN, model shapes, gradient routing, inference
import numpy as np, torch
from dataset import PairDataset
from model import KANLoFTR, homography_targets, coarse_matching_loss, kan_offset_loss
from kan import KAN

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

kan = KAN([512, 64, 2], grid_size=5, spline_order=3).to(dev)
y = kan(torch.randn(10, 512, device=dev))
assert y.shape == (10, 2) and torch.isfinite(y).all(), "KAN forward failed"

model = KANLoFTR(pretrained=None).to(dev)
print("coarse feature dim:", model.coarse_dim, "-> KAN input", model.kan.in_features)
ds = PairDataset(DATA, "train", repeat=1)
b = {k: v[None].to(dev) for k, v in ds[0].items() if k != "index"}

model.train()
enc = model.encode(b["image0"], b["image1"], b["mask0"], b["mask1"])
j, ok, p1 = homography_targets(b["H_gt"], enc)
l_coarse = coarse_matching_loss(enc, j, ok)
l_kan, rmse, base = kan_offset_loss(model, enc, j, ok, p1)
(l_coarse + l_kan).backward()
g = lambda params: sum(p.grad.abs().sum().item() for p in params if p.grad is not None)
print(f"loss coarse {l_coarse.item():.3f} | kan {l_kan.item():.3f} | rmse {rmse:.2f}px vs no-offset {base:.2f}px | valid cell pairs {int(ok.sum())}")
print(f"grad: coarse attention {g(model.loftr_coarse.parameters()):.3g} | KAN {g(model.kan.parameters()):.3g} | backbone {g(model.backbone.parameters()):.3g}")
assert g(model.loftr_coarse.parameters()) > 0 and g(model.kan.parameters()) > 0, "gradients do not reach the trainable parts"
assert g(model.backbone.parameters()) == 0, "backbone is not frozen"

model.eval()
with torch.no_grad():
    o = model(b["image0"], b["image1"], b["mask0"], b["mask1"], thr=0.0)
print("inference OK:", {k: tuple(v.shape) for k, v in o.items()})
del model, kan, enc
print("SMOKE TEST PASSED")

In [ ]:
# 5. Pretrained LoFTR weights. Offline option: attach a dataset containing loftr_outdoor.ckpt and it is copied to the torch hub cache.
ck = next(iter(INPUT.rglob("loftr_outdoor.ckpt")), None)
if ck is not None:
    dst = Path(torch.hub.get_dir()) / "checkpoints"
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(ck, dst / ck.name)
    print("using attached checkpoint:", ck)
try:
    KANLoFTR(pretrained="outdoor")
    print("pretrained LoFTR 'outdoor' weights loaded OK")
except Exception as e:
    print("COULD NOT LOAD PRETRAINED WEIGHTS:", repr(e)[:400])
    print("Turn Internet ON in the notebook settings, or attach a dataset that contains loftr_outdoor.ckpt and re-run this cell.")
    raise

In [ ]:
# 6. Fine-tune (coarse attention lr 1e-4; KAN lr 1e-3, wd 1e-2; AMP on). Best-val checkpoint -> OUT/checkpoint.pt
!python {SRC}/train.py --data {DATA} --out {OUT} --epochs 30 --batch-size 8 --repeat 20 --pretrained outdoor

In [ ]:
# 7. Training curves
import json, matplotlib.pyplot as plt
hist = json.loads((OUT / "history.json").read_text())
ep = [h["epoch"] for h in hist]
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(ep, [h["train"]["loss"] for h in hist], label="train"); ax[0].plot(ep, [h["val"]["loss"] for h in hist], label="val"); ax[0].set_title("total loss"); ax[0].legend()
ax[1].plot(ep, [h["val"]["kan_rmse_px"] for h in hist], label="KAN-refined"); ax[1].plot(ep, [h["val"]["no_offset_rmse_px"] for h in hist], "--", label="coarse centre only"); ax[1].set_title("val keypoint RMSE (px) on GT cell pairs"); ax[1].legend()
ax[2].plot(ep, [h["val"]["coarse_top1"] for h in hist]); ax[2].set_title("val coarse top-1 match accuracy")
for a in ax: a.set_xlabel("epoch")
plt.tight_layout(); plt.show()

In [ ]:
# 8. MAGSAC++ evaluation. Compare the fine-tuned model with the untouched pretrained LoFTR (the last run has no checkpoint, so only its 'coarse' column is meaningful).
!python {SRC}/evaluate.py --data {DATA} --checkpoint {OUT}/checkpoint.pt --split val  --out {WORK}/eval
!python {SRC}/evaluate.py --data {DATA} --checkpoint {OUT}/checkpoint.pt --split test --out {WORK}/eval
!python {SRC}/evaluate.py --data {DATA} --split test --out {WORK}/eval_pretrained_baseline

In [ ]:
# 9. Visual check: matches on one held-out pair; colour = error against the ground-truth homography
import cv2, matplotlib.pyplot as plt
ckpt = torch.load(OUT / "checkpoint.pt", map_location="cpu")
model = KANLoFTR(pretrained="outdoor")
model.loftr_coarse.load_state_dict(ckpt["coarse"]); model.kan.load_state_dict(ckpt["kan"])
model = model.to(dev).eval()
s = PairDataset(DATA, "val", photometric=0.0, deterministic=True, seed=7).get_numpy(0)
t = {k: torch.from_numpy(s[k])[None].to(dev) for k in ("image0", "image1", "mask0", "mask1")}
with torch.no_grad():
    o = model(**t)
kp0, kp1 = o["keypoints0"].cpu().numpy(), o["keypoints1"].cpu().numpy()
if len(kp0) == 0:
    print("no matches above the confidence threshold on this pair")
else:
    gt = cv2.perspectiveTransform(kp0.reshape(-1, 1, 2).astype(np.float64), s["H_gt"].astype(np.float64)).reshape(-1, 2)
    err = np.linalg.norm(kp1 - gt, axis=1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 5))
    ax[0].imshow(s["image0"][0], cmap="gray"); ax[0].scatter(kp0[:, 0], kp0[:, 1], s=10, c="r"); ax[0].set_title("image 0 (OHRC/NAC side)")
    ax[1].imshow(s["image1"][0], cmap="gray"); sc = ax[1].scatter(kp1[:, 0], kp1[:, 1], s=10, c=err, cmap="viridis"); ax[1].set_title("image 1 (TMC, warped)")
    plt.colorbar(sc, ax=ax[1], label="error vs H_gt (px)"); plt.tight_layout(); plt.show()
    print(f"{len(kp0)} matches | median error {np.median(err):.2f}px | 90th percentile {np.percentile(err, 90):.2f}px")

In [ ]:
# 10. What to download from the Output tab
for p in sorted(WORK.rglob("*")):
    if p.is_file() and "src" not in p.parts:
        print(f"{p.stat().st_size / 1e6:8.2f} MB  {p.relative_to(WORK)}")